# 04. Word2Vec 기반 seed dictionary 확장

`data/raw/seed_dictionary.csv`의 seed 단어를 출발점으로 삼아, DART 사업보고서 corpus에서 Word2Vec을 학습하고 각 seed와 cosine similarity가 높은 후보 단어를 뽑습니다.

이 노트북의 목적은 최종 확장 사전을 자동 확정하는 것이 아니라, 사람이 검토할 후보 단어 목록을 만드는 것입니다.

## 1. 라이브러리와 경로 설정

In [2]:
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from kiwipiepy import Kiwi

try:
    from gensim.models import Word2Vec
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "gensim이 설치되어 있지 않습니다. 터미널에서 `uv add gensim` 실행 후 커널을 재시작하세요."
    ) from exc

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

MORPH_READY_PATH = PROJECT_ROOT / "data/processed/dart_morph_ready.csv"
SEED_DICTIONARY_PATH = PROJECT_ROOT / "data/raw/seed_dictionary.csv"
STOPWORDS_PATH = PROJECT_ROOT / "data/raw/stopwords_ko_esg.txt"
OUTPUT_DIR = PROJECT_ROOT / "data/processed/seed_expansion"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MORPH_READY_PATH, SEED_DICTIONARY_PATH, OUTPUT_DIR

(PosixPath('/Users/home1/Documents/esg-dart-project/data/processed/dart_morph_ready.csv'),
 PosixPath('/Users/home1/Documents/esg-dart-project/data/raw/seed_dictionary.csv'),
 PosixPath('/Users/home1/Documents/esg-dart-project/data/processed/seed_expansion'))

## 2. 형태소 분석 결과와 seed dictionary 불러오기

In [3]:
df = pd.read_csv(MORPH_READY_PATH)
seed_dictionary = pd.read_csv(SEED_DICTIONARY_PATH)

with STOPWORDS_PATH.open("r", encoding="utf-8") as f:
    stopwords = {
        unicodedata.normalize("NFKC", line.strip()).lower()
        for line in f
        if line.strip() and not line.strip().startswith("#")
    }

print(df.shape)
display(df[["file_name", "company_name", "stock_code", "fiscal_year", "token_count"]].head())
display(seed_dictionary[["dimension", "seed_term", "pattern"]])

(381, 21)


,file_name,company_name,stock_code,fiscal_year,token_count
0,000020_2022.json,동화약품,20,2022,11845
1,000020_2023.json,동화약품,20,2023,11838
2,000020_2024.json,동화약품,20,2024,12144
3,000040_2022.json,KR모터스,40,2022,6970
4,000040_2023.json,KR모터스,40,2023,7581


,dimension,seed_term,pattern
0,E,탄소,탄소
1,E,온실가스,온실가스|GHG
2,E,탄소중립,탄소중립
3,E,넷제로,넷제로|net zero|net-zero
4,E,재생에너지,재생에너지|renewable energy
5,E,에너지,에너지
6,E,전력,전력|전력사용량
7,E,폐기물,폐기물
8,E,재활용,재활용|자원순환
9,E,폐수,폐수|수질|물관리


## 3. Word2Vec 학습용 문장 만들기

`03_전처리.ipynb`에서 만든 `tokens` 컬럼은 `단어/품사` 형태입니다. 여기서는 명사, 동사, 형용사, 영문 토큰만 남기고 Word2Vec 학습 문장으로 사용합니다.

In [4]:
KEEP_POS_PREFIXES = ("N", "V")
KEEP_POS_TAGS = {"SL"}


def normalize_word(word: str) -> str:
    return unicodedata.normalize("NFKC", str(word)).lower().strip()


def parse_tagged_tokens(tagged_text: str) -> list[str]:
    if not isinstance(tagged_text, str) or not tagged_text.strip():
        return []

    words = []
    for tagged_token in tagged_text.split():
        if "/" not in tagged_token:
            continue

        word, tag = tagged_token.rsplit("/", 1)
        word = normalize_word(word)

        if len(word) <= 1:
            continue
        if word in stopwords:
            continue
        if not re.fullmatch(r"[가-힣A-Za-z]+", word):
            continue
        if not (tag.startswith(KEEP_POS_PREFIXES) or tag in KEEP_POS_TAGS):
            continue

        words.append(word)

    return words


sentences = df["tokens"].map(parse_tagged_tokens).tolist()
sentences = [tokens for tokens in sentences if tokens]

token_lengths = pd.Series([len(tokens) for tokens in sentences])
print("학습 문서 수:", len(sentences))
print("총 토큰 수:", int(token_lengths.sum()))
display(token_lengths.describe().round(2))
sentences[0][:50]

학습 문서 수: 376
총 토큰 수: 4569239


count      376.00
mean     12152.23
std       9898.37
min       1499.00
25%       6169.75
50%       9277.50
75%      14103.25
max      55886.00
dtype: float64

['ii',
 '내용',
 '개요',
 '일반',
 '사항',
 '지배',
 '기업',
 '연결',
 '실체',
 '재화',
 '용역',
 '근거',
 '영업',
 '부문',
 '구분',
 '부문',
 '재무',
 '정보',
 '내부',
 '목적',
 '활용',
 '제약',
 '의료',
 '기기',
 '제조',
 '판매업',
 '기타',
 '금융업',
 '영위',
 '영업',
 '부문',
 '기타',
 '금융업',
 '경우',
 '중요',
 '관점',
 '비중',
 '지배',
 '현황',
 '영업',
 '개황',
 '부문',
 '구분',
 '연결',
 '연결',
 '실체',
 '매출액',
 '백만',
 '연결',
 '전년']

## 4. Word2Vec 학습

일단 후보 추출용 시작값으로 skip-gram(`sg=1`), `vector_size=100`, `window=5`, `min_count=2`, `epochs=30`을 사용합니다. 추후 후보가 너무 넓거나 좁으면 `min_count`, `window`, `epochs`, similarity threshold를 조정합니다.

In [5]:
w2v = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=1,
    seed=42,
    sg=1,
    epochs=30,
)

print("Word2Vec vocabulary size:", len(w2v.wv.key_to_index))
list(w2v.wv.key_to_index)[:50]

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

Word2Vec vocabulary size: 31298


['찬성',
 '이사',
 '자산',
 '사항',
 '금융',
 '시장',
 '위험',
 '개발',
 '위원회',
 '대하',
 '위하',
 '기타',
 '따르',
 '영업',
 '감사',
 '연결',
 '제기',
 '부문',
 '투자',
 '단위',
 '주요',
 '사외',
 '보고',
 '가결',
 '계약',
 '상품',
 '당기',
 '제품',
 '통하',
 '기업',
 '주주',
 '산업',
 '부채',
 '증가',
 '판매',
 '거래',
 '변동',
 '금액',
 '경영',
 '생산',
 '승인',
 '현황',
 '백만',
 '선임',
 '구분',
 '유동',
 '회계',
 '현금',
 '매출',
 '이사회']

## 5. seed 단어를 Word2Vec vocabulary에 매칭하기

`seed_term`과 `pattern`의 각 표현을 Kiwi로 다시 토큰화하고, Word2Vec vocabulary에 실제 존재하는 단어만 seed query로 사용합니다. 예를 들어 `net zero`는 `net`, `zero`처럼 나뉠 수 있습니다.

In [6]:
kiwi = Kiwi()


def tokenize_seed_phrase(phrase: str) -> list[str]:
    if not isinstance(phrase, str) or not phrase.strip():
        return []

    phrase = normalize_word(phrase)
    candidates = []

    # 공백/하이픈을 제거한 원 표현이 vocabulary에 있으면 우선 후보로 둡니다.
    compact = re.sub(r"[^가-힣A-Za-z]+", "", phrase)
    if len(compact) > 1:
        candidates.append(compact)

    for token in kiwi.tokenize(phrase):
        word = normalize_word(token.form)
        if len(word) <= 1:
            continue
        if not re.fullmatch(r"[가-힣A-Za-z]+", word):
            continue
        if not (token.tag.startswith(KEEP_POS_PREFIXES) or token.tag in KEEP_POS_TAGS):
            continue
        candidates.append(word)

    return sorted(set(candidates))


def seed_query_terms(row: pd.Series) -> list[str]:
    phrases = [row["seed_term"]]
    if isinstance(row.get("pattern"), str):
        phrases.extend(row["pattern"].split("|"))

    query_terms = []
    for phrase in phrases:
        query_terms.extend(tokenize_seed_phrase(phrase))

    return sorted({term for term in query_terms if term in w2v.wv})


seed_dictionary["w2v_query_terms"] = seed_dictionary.apply(seed_query_terms, axis=1)
seed_dictionary["w2v_query_count"] = seed_dictionary["w2v_query_terms"].map(len)

display(seed_dictionary[["dimension", "seed_term", "pattern", "w2v_query_terms", "w2v_query_count"]])

missing_seed = seed_dictionary.loc[seed_dictionary["w2v_query_count"] == 0, ["dimension", "seed_term", "pattern"]]
print("Word2Vec vocabulary에 매칭되지 않은 seed 수:", len(missing_seed))
display(missing_seed)

,dimension,seed_term,pattern,w2v_query_terms,w2v_query_count
0,E,탄소,탄소,[탄소],1
1,E,온실가스,온실가스|GHG,[온실가스],1
2,E,탄소중립,탄소중립,"[중립, 탄소]",2
3,E,넷제로,넷제로|net zero|net-zero,"[net, netzero, zero]",3
4,E,재생에너지,재생에너지|renewable energy,"[renewable, renewableenergy, 에너지, 재생]",4
5,E,에너지,에너지,[에너지],1
6,E,전력,전력|전력사용량,"[사용량, 전력]",2
7,E,폐기물,폐기물,[폐기물],1
8,E,재활용,재활용|자원순환,"[순환, 자원, 활용]",3
9,E,폐수,폐수|수질|물관리,"[수질, 폐수]",2


Word2Vec vocabulary에 매칭되지 않은 seed 수: 0


,dimension,seed_term,pattern


## 6. seed별 유사어 후보 추출

시작 임계값은 `SIMILARITY_THRESHOLD = 0.55`입니다. 너무 많은 잡음이 나오면 0.60~0.70으로 올리고, 후보가 너무 적으면 0.45~0.50으로 낮춰 비교합니다.

In [7]:
SIMILARITY_THRESHOLD = 0.55
TOPN_PER_QUERY_TERM = 50


def similar_words_for_seed(query_terms: list[str]) -> list[dict]:
    if not query_terms:
        return []

    best = {}

    for query_term in query_terms:
        for word, similarity in w2v.wv.most_similar(query_term, topn=TOPN_PER_QUERY_TERM):
            word = normalize_word(word)

            if word in query_terms:
                continue
            if word in stopwords:
                continue
            if not re.fullmatch(r"[가-힣A-Za-z]+", word):
                continue
            if similarity < SIMILARITY_THRESHOLD:
                continue

            if word not in best or similarity > best[word]["similarity"]:
                best[word] = {
                    "candidate": word,
                    "similarity": float(similarity),
                    "source_query_term": query_term,
                }

    return sorted(best.values(), key=lambda row: row["similarity"], reverse=True)


expanded_rows = []

for seed_row in seed_dictionary.itertuples(index=False):
    candidates = similar_words_for_seed(seed_row.w2v_query_terms)

    for rank, candidate in enumerate(candidates, start=1):
        expanded_rows.append({
            "dimension": seed_row.dimension,
            "seed_term": seed_row.seed_term,
            "query_terms": ", ".join(seed_row.w2v_query_terms),
            "rank": rank,
            "candidate": candidate["candidate"],
            "similarity": candidate["similarity"],
            "source_query_term": candidate["source_query_term"],
        })

expanded_seed_candidates = pd.DataFrame(expanded_rows)

print("후보 row 수:", len(expanded_seed_candidates))
display(expanded_seed_candidates.head(100))

후보 row 수: 1788


,dimension,seed_term,query_terms,rank,candidate,similarity,source_query_term
0,E,탄소,탄소,1,중립,0.835202,탄소
1,E,탄소,탄소,2,recylcle,0.677535,탄소
2,E,탄소,탄소,3,온실가스,0.661899,탄소
3,E,탄소,탄소,4,배출량,0.660234,탄소
4,E,탄소,탄소,5,기후,0.657483,탄소
...,...,...,...,...,...,...,...
95,E,온실가스,온실가스,46,저감,0.574610,온실가스
96,E,온실가스,온실가스,47,산성비,0.574189,온실가스
97,E,온실가스,온실가스,48,esgreport,0.569645,온실가스
98,E,온실가스,온실가스,49,야심,0.568860,온실가스


## 7. 각 seed별 후보 목록 출력

In [8]:
MAX_PRINT_PER_SEED = 30

for seed_row in seed_dictionary.itertuples(index=False):
    sub = expanded_seed_candidates.loc[
        (expanded_seed_candidates["dimension"] == seed_row.dimension)
        & (expanded_seed_candidates["seed_term"] == seed_row.seed_term)
    ].head(MAX_PRINT_PER_SEED)

    print("=" * 100)
    print(f"[{seed_row.dimension}] {seed_row.seed_term}")
    print("query_terms:", ", ".join(seed_row.w2v_query_terms) if seed_row.w2v_query_terms else "매칭 없음")

    if sub.empty:
        print("후보 없음")
        continue

    for row in sub.itertuples(index=False):
        print(f"{row.rank:02d}. {row.candidate}\t{row.similarity:.4f}\t(source={row.source_query_term})")

[E] 탄소
query_terms: 탄소
01. 중립	0.8352	(source=탄소)
02. recylcle	0.6775	(source=탄소)
03. 온실가스	0.6619	(source=탄소)
04. 배출량	0.6602	(source=탄소)
05. 기후	0.6575	(source=탄소)
06. 이산화	0.6543	(source=탄소)
07. euetseu	0.6470	(source=탄소)
08. 로드맵	0.6397	(source=탄소)
09. sbtisciencebasedtargetsinitiative	0.6397	(source=탄소)
10. 발자국	0.6382	(source=탄소)
11. biorecycle	0.6363	(source=탄소)
12. membrane	0.6339	(source=탄소)
13. mitigation	0.6321	(source=탄소)
14. netzero	0.6305	(source=탄소)
15. 저감	0.6296	(source=탄소)
16. activegreeninitiative	0.6279	(source=탄소)
17. iaa	0.6272	(source=탄소)
18. 임목	0.6234	(source=탄소)
19. expo	0.6209	(source=탄소)
20. sbti	0.6203	(source=탄소)
21. scope	0.6185	(source=탄소)
22. zerowaste	0.6183	(source=탄소)
23. netzeroroadmap	0.6172	(source=탄소)
24. 배출	0.6165	(source=탄소)
25. co	0.6160	(source=탄소)
26. carbonfree	0.6140	(source=탄소)
27. nccpo	0.6076	(source=탄소)
28. 녹색	0.6072	(source=탄소)
29. cii	0.6029	(source=탄소)
30. zerocarbondrive	0.6005	(source=탄소)
[E] 온실가스
query_terms: 온실가스
01. 배출량	0.9155	(source=온

## 8. 결과 저장

저장된 CSV는 사람이 후보를 검토하면서 `keep/drop`, `reason` 같은 컬럼을 추가해 최종 expanded dictionary 초안으로 사용할 수 있습니다.

In [9]:
threshold_label = str(SIMILARITY_THRESHOLD).replace(".", "p")
candidate_path = OUTPUT_DIR / f"w2v_seed_candidates_threshold_{threshold_label}.csv"
query_match_path = OUTPUT_DIR / "w2v_seed_query_terms.csv"

expanded_seed_candidates.to_csv(candidate_path, index=False, encoding="utf-8-sig")
seed_dictionary[["dimension", "seed_term", "pattern", "w2v_query_terms", "w2v_query_count"]].to_csv(
    query_match_path,
    index=False,
    encoding="utf-8-sig",
)

print("후보 저장:", candidate_path)
print("seed-query 매칭 저장:", query_match_path)

후보 저장: /Users/home1/Documents/esg-dart-project/data/processed/seed_expansion/w2v_seed_candidates_threshold_0p55.csv
seed-query 매칭 저장: /Users/home1/Documents/esg-dart-project/data/processed/seed_expansion/w2v_seed_query_terms.csv


## 9. CSV 결과 정리와 기술적 진행 방식

이 노트북은 `seed_dictionary.csv`의 30개 ESG seed 단어를 출발점으로 삼아, DART 사업보고서 corpus 안에서 Word2Vec 유사어 후보를 만든다. 여기서 생성한 후보는 최종 expanded dictionary가 아니라, 사람이 직접 검토할 1차 후보 목록이다.

### 생성된 CSV

실행 결과는 `data/processed/seed_expansion/` 아래에 저장된다.

| 파일 | 내용 |
|---|---|
| `w2v_seed_candidates_threshold_0p55.csv` | seed별 Word2Vec 유사어 후보 목록 |
| `w2v_seed_query_terms.csv` | 각 seed가 Word2Vec vocabulary에서 어떤 query term으로 매칭되었는지 정리한 표 |

`w2v_seed_candidates_threshold_0p55.csv`의 주요 컬럼은 다음과 같다.

| 컬럼 | 의미 |
|---|---|
| `dimension` | seed의 ESG 차원. `E`, `S`, `G` |
| `seed_term` | 원래 seed dictionary의 seed 단어 |
| `query_terms` | Word2Vec vocabulary에 실제 매칭된 seed 표현 |
| `rank` | 해당 seed 안에서 similarity 기준 후보 순위 |
| `candidate` | 확장 후보 단어 |
| `similarity` | seed query term과 candidate 사이의 cosine similarity |
| `source_query_term` | candidate를 끌어낸 기준 query term |

`w2v_seed_query_terms.csv`는 seed가 모델 vocabulary에 잘 잡혔는지 확인하는 용도다. `w2v_query_count`가 0이면 해당 seed는 Word2Vec 모델에서 직접 유사어를 뽑지 못한 것이다.

### 기술적 진행 방식

전체 절차는 아래 순서로 진행했다.

```text
03_전처리 결과 dart_morph_ready.csv 로드
→ tokens 컬럼에서 단어/품사 분리
→ 명사, 동사/형용사, 영문 토큰만 유지
→ stopwords, 특수문자, 한 글자 토큰 제거
→ 기업-연도별 토큰 리스트를 Word2Vec 학습 문장으로 사용
→ seed_term과 pattern을 같은 기준으로 토큰화
→ Word2Vec vocabulary에 존재하는 seed 표현만 query term으로 사용
→ 각 query term의 most_similar 후보 추출
→ similarity threshold 이상 후보만 남김
→ seed별 후보 CSV 저장
```

Word2Vec 학습 설정은 다음과 같다.

```python
Word2Vec(
    vector_size=100,
    window=5,
    min_count=2,
    workers=1,
    seed=42,
    sg=1,
    epochs=30,
)
```

여기서 `sg=1`은 skip-gram 방식이다. skip-gram은 비교적 드문 단어의 주변 문맥을 학습하는 데 유리해서, ESG seed처럼 특정 주제어의 후보를 넓히는 목적에 맞다.

### 임계값 설정

현재 유사도 임계값은 시작값으로 다음을 사용했다.

```python
SIMILARITY_THRESHOLD = 0.55
```

이 값은 최종 기준이 아니라 탐색용 기준이다. 후보가 너무 많고 잡음이 많으면 `0.60` 또는 `0.70`으로 올리고, 후보가 너무 적으면 `0.45` 또는 `0.50`으로 낮춰 비교한다.

보고서에는 한 임계값만 제시하기보다, 몇 가지 threshold를 비교한 뒤 최종 선택 이유를 쓰는 것이 좋다.

```text
threshold가 낮음 → 후보 수 증가, 잡음 증가
threshold가 높음 → 후보 수 감소, 관련 단어 누락 가능성 증가
```

### 사람이 검토해야 하는 이유

Word2Vec 유사어는 의미상 ESG와 가까운 단어뿐 아니라, 같은 문맥에 자주 등장하는 회사명, 산업명, 회계/공시 일반어도 함께 끌어올 수 있다. 따라서 CSV 결과를 그대로 expanded dictionary로 쓰면 안 된다.

검토 시 우선 제외할 후보는 다음과 같다.

| 제외 후보 유형 | 예시 |
|---|---|
| 회사명/브랜드명 | 특정 기업명, 제품명 |
| 너무 일반적인 경영 단어 | 사업, 회사, 관리, 추진 등 |
| 숫자/단위에서 파생된 표현 | 백만원, 기말 등 |
| ESG 의미가 약한 산업 일반어 | 업종별 반복 명사 |
| seed와 문맥만 가깝고 의미는 다른 단어 | 표 제목, 공시 양식 표현 |

반대로 유지할 후보는 다음 조건을 만족하는 단어다.

```text
1. seed와 같은 E/S/G 차원의 의미를 가진다.
2. 실제 DART 문장 안에서 ESG 표현으로 해석 가능하다.
3. 너무 특정 기업에만 종속된 고유명사가 아니다.
4. 회귀 feature로 합산했을 때 해석 가능한 단어다.
```

### 다음 단계

다음 단계에서는 `w2v_seed_candidates_threshold_0p55.csv`를 열어 후보별로 `keep`, `drop`, `reason` 컬럼을 추가해 사람이 직접 필터링한다. 이후 유지된 후보만 모아 `expanded_dictionary.csv`를 만들고, `03_전처리.ipynb`의 seed TF-IDF 계산 방식과 동일하게 expanded TF-IDF score를 계산하면 된다.

최종 보고서에는 다음 내용을 남긴다.

```text
- Word2Vec 학습 corpus: DART II, IV, VI 섹션 기반 기업-연도 문서
- 전처리 기준: Kiwi 형태소 분석, stopwords 제거, 명사/동사/형용사/영문 유지
- Word2Vec 설정: vector_size, window, min_count, sg, epochs
- threshold 후보 비교와 최종 선택 이유
- 사람이 제거한 잡음 유형
- 최종 expanded dictionary 크기와 E/S/G별 후보 수
```



제안하신 방법의 핵심
기준 ESG 문장 3개(E, S, G 각각)를 dense vector로 만들고, 분석 대상 문서의 모든 문장도 dense vector로 만든 뒤, cosine similarity로 ESG 관련 문장만 필터링 → 그 위에서 비율 같은 더 정교한 피처를 뽑겠다